# 🚀 Full Training Pipeline Sparse MoE 400M-A188M di Colab A100
Notebook resmi untuk pelatihan **Liwet-400M-A188M Sparse MoE (`IndonesianChatLM`)** secara end-to-end.

### 🧠 Arsitektur Sparse Mixture-of-Experts:
- **Total Parameter:** **400.1M** | **Parameter Aktif:** **187.8M** per token
- **Expert Configuration:** 8 Routed Experts (Top-2 routing)
- **Router Load Balancing:** Auxiliary load-balancing loss + z-loss + DeepSeek-style EMA bias update
- **Akselerasi A100:** `torch.compile` mode `max-autotune-no-cudagraphs`, SDPA FlashAttention, bfloat16, length-bucketed batching (bebas padding)

### 🔄 Tahapan Pipeline (Otomatis & Terhubung):
1. **Pretraining MoE 400M** (~2.1B Tokens / 32.000 step @ ~80k-90k tok/s)
2. **SFT MoE 400M** (165.640 dialog percakapan Indonesia + MoE aux loss)
3. **SimPO Preference Alignment MoE 400M** (8.212 pasangan preference + router balancing)
4. **Daemon Auto-Sync Drive:** Sinkronisasi background tiap 60 detik (GPU bebas freeze, aman disconnect)
5. **Uji Generasi Interaktif & Visualisasi Kurva Loss / Router Balance**
**Paths:** set `DRIVE_MOUNT` and `WORK_DIR` environment variables before running; this notebook stores no machine-specific absolute paths.


In [ ]:
# === 1. Pemeriksaan Hardware GPU A100 ===
!nvidia-smi
import torch
assert torch.cuda.is_available(), '❌ GPU tidak terdeteksi! Klik Menu Runtime > Change runtime type > A100 GPU.'
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
bf16_ok = torch.cuda.is_bf16_supported()
print(f'✅ GPU: {gpu_name} | VRAM: {vram_gb:.1f} GB | BF16 Supported: {bf16_ok} | PyTorch: {torch.__version__}')

In [ ]:
# === 2. Mount Google Drive ===
from pathlib import Path
from google.colab import drive

import os
DRIVE_MOUNT = Path(os.environ.get('DRIVE_MOUNT', 'drive'))
if not (DRIVE_MOUNT / 'MyDrive').exists():
    drive.mount(str(DRIVE_MOUNT))

DRIVE_ROOT = DRIVE_MOUNT / 'MyDrive' / 'Transformer_400M_Colab'
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print(f'✅ Google Drive terhubung di: {DRIVE_ROOT}')

In [ ]:
# === 3. Setup Direktori Kerja NVMe Lokal (Super Cepat) ===
import os, shutil
from pathlib import Path

WORK_DIR = Path(os.environ.get('WORK_DIR', 'work'))
PROJECT_ROOT = WORK_DIR / 'TUBES_1_400M_MOE'
DATA_DIR = PROJECT_ROOT / 'data'
CHECKPOINT_DIR = PROJECT_ROOT / 'checkpoints'

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print('✅ Direktori NVMe SSD siap:', PROJECT_ROOT)

In [ ]:
# === 4. Ekstraksi Kode MoE & Dataset ke NVMe ===
import tarfile, time, shutil
from pathlib import Path

t0 = time.time()
# 1. Ekstrak Kode MoE 400M
code_tar = DRIVE_ROOT / '01_code_moe_400m.tar'
if not code_tar.exists():
    raise FileNotFoundError(f'{code_tar} belum ada di Drive!')
print(f'Mengekstrak kode dari {code_tar.name} ({code_tar.stat().st_size/1e6:.2f} MB) ...')
shutil.rmtree(PROJECT_ROOT / 'model', ignore_errors=True)
shutil.rmtree(PROJECT_ROOT / 'train', ignore_errors=True)
with tarfile.open(code_tar, 'r:*') as tf:
    tf.extractall(PROJECT_ROOT)
print('✅ Kode MoE 400M berhasil diekstrak!')

# 2. Tangani Dataset Pretrain Tokenized
tok_dir = DATA_DIR / 'tokenized'
tok_dir.mkdir(parents=True, exist_ok=True)

# Jika sebelumnya sempat terekstrak ke subfolder bersarang data/tokenized/data/tokenized
nested_tok = tok_dir / 'data' / 'tokenized'
if nested_tok.exists() and (nested_tok / 'index.json').exists():
    print(f'Merapikan dataset tokenized dari subfolder bersarang {nested_tok} ...')
    for item in nested_tok.iterdir():
        dst = tok_dir / item.name
        if dst.exists():
            if dst.is_dir():
                shutil.rmtree(dst)
            else:
                dst.unlink()
        shutil.move(str(item), str(dst))
    shutil.rmtree(tok_dir / 'data', ignore_errors=True)

# Jika index.json belum ada di tok_dir, ekstrak dari tar ke PROJECT_ROOT
if not (tok_dir / 'index.json').exists():
    for tar_name in ['03_tokenized_part1.tar', '04_tokenized_part2.tar']:
        p = DRIVE_ROOT / tar_name
        if p.exists():
            print(f'Mengekstrak {tar_name} ({p.stat().st_size/1e9:.2f} GB) langsung ke {PROJECT_ROOT} ...')
            with tarfile.open(p, 'r:*') as tf:
                tf.extractall(PROJECT_ROOT)
            print(f'Selesai ekstrak {tar_name}')

# 3. Ekstrak Dataset SFT & Preference (Post-Training)
sft_ok = (DATA_DIR / 'sft' / 'train.jsonl').exists()
pref_ok = (DATA_DIR / 'preference' / 'train.jsonl').exists()

if not (sft_ok and pref_ok):
    sft_cands = [
        DRIVE_ROOT / '02_dataset_sft_preference.tar',
        DRIVE_MOUNT / 'MyDrive' / 'Transformer_163M_Colab' / '01_code_sft_pref_tokenizer.tar',
        DRIVE_MOUNT / 'MyDrive' / 'colab_bundle' / '01_code_sft_pref_tokenizer.tar',
    ]
    sft_tar_found = next((p for p in sft_cands if p.exists() and p.stat().st_size > 1024), None)
    if sft_tar_found:
        print(f'Mengekstrak dataset SFT & Preference dari {sft_tar_found.name} ...')
        with tarfile.open(sft_tar_found, 'r:*') as tf:
            for m in tf.getmembers():
                if m.name.startswith(('data/sft', 'data/preference', 'data/tokenizer')):
                    tf.extract(m, PROJECT_ROOT)
        print('✅ Dataset SFT & Preference siap!')
    else:
        print('⚠️ Archive SFT/Preference belum ditemukan, memeriksa folder data lokal...')

# 4. Validasi Kelengkapan Total
assert (tok_dir / 'index.json').exists(), f'❌ index.json tidak ditemukan di {tok_dir}!'
train_shards = list((tok_dir / 'train').glob('*.bin'))
val_shards = list((tok_dir / 'val').glob('*.bin'))
assert len(train_shards) >= 20, f'❌ Shard pretrain kurang! Ditemukan {len(train_shards)} shard.'
assert (DATA_DIR / 'sft' / 'train.jsonl').exists(), f'❌ SFT train.jsonl belum ada di {DATA_DIR / "sft"}!'
assert (DATA_DIR / 'preference' / 'train.jsonl').exists(), f'❌ Preference train.jsonl belum ada di {DATA_DIR / "preference"}!'

print('=' * 65)
print('✅ SEMUA DATASET SIAP 100%:')
print(f'   - Pretrain Train Shards : {len(train_shards)} bin files')
print(f'   - Pretrain Val Shards   : {len(val_shards)} bin files')
print(f'   - Pretrain Index        : {tok_dir / "index.json"}')
print(f'   - SFT Dataset           : {DATA_DIR / "sft" / "train.jsonl"}')
print(f'   - Preference Dataset    : {DATA_DIR / "preference" / "train.jsonl"}')
print(f'Ekstraksi & validasi tuntas dalam {time.time()-t0:.1f} detik!')
print('=' * 65)


In [ ]:
# === 5. Verifikasi Arsitektur MoE 400M & Parameter Count ===
import sys, importlib, torch
# Bersihkan cache import lama jika runtime sempat me-load versi dense
for mod in list(sys.modules.keys()):
    if mod in ['model', 'model.model', 'train_utils']:
        del sys.modules[mod]

sys.path.insert(0, str(PROJECT_ROOT / 'train'))
sys.path.insert(0, str(PROJECT_ROOT / 'model'))
sys.path.insert(0, str(PROJECT_ROOT))

from model import IndonesianChatLM, ModelConfig
try:
    from model import count_active_parameters
except ImportError:
    from model.model import count_active_parameters
from train_utils import count_parameters, model_summary

config = ModelConfig.from_json(PROJECT_ROOT / 'model' / 'config.json')
print('Config Model:', config.to_dict())

# Cek instansiasi model ringan di meta/cpu
with torch.device('meta'):
    test_model = IndonesianChatLM(config)
    total_params = count_parameters(test_model)
    active_params = count_active_parameters(test_model)

print('=' * 60)
print(f'🔥 TOTAL PARAMETER : {total_params:,} ({total_params/1e6:.1f}M)')
print(f'⚡ AKTIF PER TOKEN : {active_params:,} ({active_params/1e6:.1f}M)')
print(f'🎯 JUMLAH EXPERTS  : {config.num_experts} (Top-{config.num_experts_per_tok} routed)')
print(f'⚖️ ROUTER BALANCE  : Aux Loss ({config.router_aux_loss_coef}) + Z-Loss ({config.router_z_loss_coef}) + EMA Bias')
print('=' * 60)

In [ ]:
# === 6. Background Syncer Daemon (Safe & Atomic NVMe -> Drive) ===
# Menyinkronkan step_* DAN final yang SUDAH selesai ditulis (lolos validasi biner)
import shutil, threading, time
from pathlib import Path

DRIVE_CKPT_ROOT = DRIVE_ROOT / 'checkpoints'
DRIVE_CKPT_ROOT.mkdir(parents=True, exist_ok=True)
sync_stop_signal = False

def is_valid_checkpoint(p: Path) -> bool:
    pt = p / 'pytorch_model.bin'
    st = p / 'training_state.pt'
    cfg = p / 'config.json'
    if not pt.exists() or not cfg.exists() or pt.stat().st_size < 1.5e9:
        return False
    if p.name != 'final' and (not st.exists() or st.stat().st_size < 3.0e9):
        return False
    if p.name == 'final' and st.exists() and st.stat().st_size < 3.0e9:
        return False
    try:
        with open(pt, 'rb') as f:
            f.seek(max(0, pt.stat().st_size - 65536))
            if b'PK\x05\x06' not in f.read():
                return False
        if st.exists():
            with open(st, 'rb') as f:
                f.seek(max(0, st.stat().st_size - 65536))
                if b'PK\x05\x06' not in f.read():
                    return False
    except Exception:
        return False
    return True

def safe_drive_sync_worker(local_root: Path, drive_root: Path, interval: int = 30):
    print(f'[Safe Syncer] Daemon aktif (interval {interval}s). Mem-backup step_* dan final yang 100% utuh.')
    while not sync_stop_signal:
        time.sleep(interval)
        try:
            for stage in ['pretrain', 'sft', 'simpo']:
                local_stage = local_root / stage
                drive_stage = drive_root / stage
                if not local_stage.exists():
                    continue
                drive_stage.mkdir(parents=True, exist_ok=True)
                for fname in ['metrics.jsonl', 'train.log', 'trainer_args.json']:
                    fsrc, fdst = local_stage / fname, drive_stage / fname
                    if fsrc.exists():
                        shutil.copy2(fsrc, fdst)
                candidates = list(local_stage.glob('step_*'))
                if (local_stage / 'final').exists():
                    candidates.append(local_stage / 'final')
                for ckpt in candidates:
                    if is_valid_checkpoint(ckpt):
                        target = drive_stage / ckpt.name
                        if not is_valid_checkpoint(target):
                            print(f'\n[Safe Syncer] Backup checkpoint utuh {stage}/{ckpt.name} ke Drive...')
                            shutil.copytree(ckpt, target, dirs_exist_ok=True)
        except Exception:
            pass
    print('[Safe Syncer] Daemon selesai.')

sync_thread = threading.Thread(target=safe_drive_sync_worker, args=(CHECKPOINT_DIR, DRIVE_CKPT_ROOT, 30), daemon=True)
sync_thread.start()
print('✅ Safe Background syncer aktif!')

In [ ]:
# === OTOMASI PEMULIHAN CHECKPOINT DARI GOOGLE DRIVE KE NVME LOKAL ===
# Berfungsi otomatis saat mesin baru di-restart / disconnect
import json, shutil
from pathlib import Path

DRIVE_CKPT = DRIVE_ROOT / 'checkpoints' / 'pretrain'
LOCAL_CKPT = CHECKPOINT_DIR / 'pretrain'
LOCAL_CKPT.mkdir(parents=True, exist_ok=True)

# 1. Bersihkan checkpoint lokal yang tidak lengkap / corrupt jika ada
for p in LOCAL_CKPT.glob('step_*'):
    if not is_valid_checkpoint(p):
        print(f'⚠️ Membersihkan checkpoint lokal terpotong: {p.name}')
        shutil.rmtree(p, ignore_errors=True)

# 2. Pulihkan riwayat log dan metrik sebelumnya jika ada di Drive
if DRIVE_CKPT.exists():
    for fname in ['metrics.jsonl', 'train.log', 'trainer_args.json']:
        src = DRIVE_CKPT / fname
        dst = LOCAL_CKPT / fname
        if src.exists() and not dst.exists():
            shutil.copy2(src, dst)

    # 3. Deteksi checkpoint valid tertinggi di Drive
    valid_steps = [p for p in DRIVE_CKPT.glob('step_*') if is_valid_checkpoint(p)]
    valid_steps.sort(key=lambda p: int(p.name.split('_')[1]))

    if valid_steps:
        latest = valid_steps[-1]
        target_local = LOCAL_CKPT / latest.name
        if not is_valid_checkpoint(target_local):
            print(f'🔄 Memulihkan checkpoint valid terbaru dari Drive: {latest.name} (4.8 GB) ke NVMe...')
            shutil.copytree(latest, target_local, dirs_exist_ok=True)
            print(f'✅ {latest.name} berhasil disalin dan lolos uji integritas!')
        else:
            print(f'✅ Checkpoint {latest.name} sudah siap dan valid di NVMe lokal.')
    else:
        print('ℹ️ Belum ada checkpoint valid di Drive, memulai training baru.')

# 4. Kunci konfigurasi autotune agar tidak mengulang benchmark
autotune_file = PROJECT_ROOT / 'autotune_a100_results.json'
if not autotune_file.exists():
    default_autotune = {
        'pretrain': {'seq_len': 1024, 'batch_size': 12, 'grad_accum': 4, 'compile_mode': 'max-autotune-no-cudagraphs'},
        'sft': {'seq_len': 2048, 'batch_size': 4, 'grad_accum': 8, 'compile_mode': 'default'},
        'simpo': {'seq_len': 1024, 'batch_size': 4, 'grad_accum': 16, 'compile_mode': 'max-autotune-no-cudagraphs'}
    }
    autotune_file.write_text(json.dumps(default_autotune, indent=2), encoding='utf-8')
    print('✅ Konfigurasi A100 optimal dimuat.')


In [ ]:
# === 7. JALANKAN FULL PIPELINE MOE 400M (Pretrain -> SFT -> SimPO) ===
# Pipeline ini otomatis menghubungkan checkpoint antar tahap secara berurutan.
# Tidak ada batas waktu (tanpa max-hours), training jalan terus sampai selesai.
import os, subprocess, sys

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
env['TORCHINDUCTOR_CACHE_DIR'] = str(WORK_DIR / 'triton_cache')

cmd = [
    sys.executable, '-u', str(PROJECT_ROOT / 'train' / 'pipeline_a100.py'),
    '--resume',
    '--save-interval-simpo', '30',
    '--eval-interval-simpo', '30',
    '--keep-checkpoints', '6',
]

print('🚀 Menjalankan Full Pipeline MoE 400M:')
print('  ', ' '.join(cmd))
proc = subprocess.Popen(cmd, cwd=str(PROJECT_ROOT), env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
rc = proc.wait()
if rc != 0:
    raise RuntimeError(f'❌ Pipeline MoE gagal dengan exit code {rc}')

print('\n🎉🎉 FULL PIPELINE MOE 400M SELESAI SUKSES 100%! 🎉🎉')

In [ ]:
# === 8. Uji Generasi Percakapan Model MoE 400M (Inference Test) ===
import sys, torch
from tokenizers import Tokenizer
sys.path.insert(0, str(PROJECT_ROOT / 'train'))
from train_utils import load_model, autocast_context
from chat_format import render_prompt, token_ids

# Deteksi model terbaru (prioritas: simpo -> sft -> pretrain)
candidate_models = [
    CHECKPOINT_DIR / 'simpo' / 'final',
    CHECKPOINT_DIR / 'sft' / 'final',
    CHECKPOINT_DIR / 'pretrain' / 'final',
]
model_dir = next((p for p in candidate_models if (p / 'pytorch_model.bin').exists()), None)
if model_dir is None:
    print('Belum ada checkpoint final yang selesai.')
else:
    print(f'Memuat model dari: {model_dir}')
    device = torch.device('cuda')
    model = load_model(model_dir, device)
    model.eval()
    tokenizer = Tokenizer.from_file(str(DATA_DIR / 'tokenizer' / 'tokenizer.json'))
    bos_id, eos_id, pad_id = token_ids(tokenizer)

    test_prompts = [
        'Halo! Siapa kamu dan apa kelebihan model Mixture-of-Experts?',
        'Jelaskan mengapa bahasa Indonesia penting dalam era kecerdasan buatan.',
        'Berikan 3 tips praktis untuk belajar pemrograman AI secara efektif.'
    ]
    for prompt in test_prompts:
        text = render_prompt([{'role': 'user', 'content': prompt}])
        ids = [bos_id] + tokenizer.encode(text, add_special_tokens=False).ids
        input_ids = torch.tensor([ids], dtype=torch.long, device=device)
        with torch.no_grad(), autocast_context(device, 'bf16'):
            out = model.generate(
                input_ids,
                max_new_tokens=150,
                temperature=0.7,
                top_k=50,
                top_p=0.9,
                repetition_penalty=1.1,
                eos_token_id=eos_id,
                pad_token_id=pad_id,
            )
        answer = tokenizer.decode(out[0, len(ids):].tolist()).strip()
        print('=' * 72)
        print('User     :', prompt)
        print('MoE 400M :', answer)
        print('=' * 72)

In [ ]:
# === 9. Visualisasi Kurva Training & Metrik Lengkap ===
import json
import matplotlib.pyplot as plt
from pathlib import Path

def load_metrics_log(metric_path):
    rows = []
    if Path(metric_path).exists():
        with open(metric_path, encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    try:
                        rows.append(json.loads(line))
                    except Exception:
                        pass
    return rows

# Muat metrik dari tiap tahap
pt_rows = load_metrics_log(CHECKPOINT_DIR / 'pretrain' / 'metrics.jsonl')
sft_rows = load_metrics_log(CHECKPOINT_DIR / 'sft' / 'metrics.jsonl')
sp_rows = load_metrics_log(CHECKPOINT_DIR / 'simpo' / 'metrics.jsonl')

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

# 1. Pretrain Loss & PPL
if pt_rows:
    steps = [r['step'] for r in pt_rows if 'train_loss' in r]
    losses = [r['train_loss'] for r in pt_rows if 'train_loss' in r]
    axes[0].plot(steps, losses, label='Pretrain Loss', color='royalblue')
    axes[0].set_title('Stage 1: Pretrain MoE 400M Loss')
    axes[0].set_xlabel('Step')
    axes[0].set_ylabel('Loss')
    axes[0].grid(alpha=0.3)
    axes[0].legend()

# 2. SFT Loss
if sft_rows:
    s_steps = [r['step'] for r in sft_rows if 'train_loss' in r]
    s_loss = [r['train_loss'] for r in sft_rows if 'train_loss' in r]
    axes[1].plot(s_steps, s_loss, label='SFT Loss', color='coral')
    axes[1].set_title('Stage 2: SFT Loss')
    axes[1].set_xlabel('Step')
    axes[1].set_ylabel('Loss')
    axes[1].grid(alpha=0.3)
    axes[1].legend()

# 3. SimPO Margin Loss / Accuracy
if sp_rows:
    sp_steps = [r['step'] for r in sp_rows if 'train_loss' in r]
    sp_loss = [r['train_loss'] for r in sp_rows if 'train_loss' in r]
    axes[2].plot(sp_steps, sp_loss, label='SimPO Loss', color='forestgreen')
    axes[2].set_title('Stage 3: SimPO Loss')
    axes[2].set_xlabel('Step')
    axes[2].set_ylabel('Loss')
    axes[2].grid(alpha=0.3)
    axes[2].legend()

plt.tight_layout()
plt.show()